In [1]:
from torch.fx.experimental.unification.unification_tools import get_in

from pydrake.all import (
    StartMeshcat,
    AddDefaultVisualization,
    Simulator,
    RobotDiagramBuilder,
    VPolytope,
    HPolyhedron,
    SceneGraphCollisionChecker,
    RandomGenerator,
    PointCloud,
    Rgba,
    Quaternion,
    RigidTransform,
    IrisFromCliqueCoverOptions,
    IrisInConfigurationSpaceFromCliqueCoverV2,
    SaveIrisRegionsYamlFile,
    LoadModelDirectives,
    ProcessModelDirectives,
    CollisionCheckerParams,
    Sphere,
    GaussianVectorX,
    RandomGenerator,
    UniformVector,
    IrisZoFromCliqueBuilder,
    MinCliqueCoverSolverViaGreedy,
    MaxCliqueSolverViaGreedy,
    Parallelism,
    PointsToCliqueCoverSets,
    VisibilityGraph,
    HPolyhedronPointSampler,
    InverseKinematics,
    Solve,
    LoadIrisRegionsYamlFile,
)
import numpy as np
import os
import data.iris_benchmarks.benchmarks.helpers as benchmark_helpers
from data.iris_benchmarks.benchmarks.helpers import load_seed_points
from data.iris_benchmarks.iris_environments.environments import get_environment_builder
import networkx as nx
import tqdm
import time
import pathlib
import pickle
import utils
import matplotlib.pyplot as plt

prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']


In [2]:
TEST_SCENE = "7DOFIIWA"
EXPERIMENT_NAME = "rrt_seeding"

src_directory = os.path.abspath(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
parent_directory = os.path.dirname(src_directory)
data_directory = os.path.join(parent_directory, "data")

experiment_data_folder = os.path.join(data_directory, TEST_SCENE, EXPERIMENT_NAME)
pathlib.Path(experiment_data_folder).mkdir(parents=True, exist_ok=True)
region_file = os.path.join(experiment_data_folder, "iris_regions.yaml")

plant, scene_graph, diagram, diagram_context, plant_context, models, meshcat =  get_environment_builder(TEST_SCENE)(True)
diagram.ForcedPublish(diagram_context)
print("Model Names:")
for i, model in enumerate(models):
    print(f"models[{i}]: {model.model_name}")
collision_checker = SceneGraphCollisionChecker(model = diagram,
                                               robot_model_instances = [model.model_instance for model in models],
                                               edge_step_size = 0.02)

INFO:drake:Meshcat listening for connections at http://localhost:7000
INFO:drake:Allocating contexts to support implicit context parallelism 16


Model Names:
models[0]: iiwa
models[1]: wsg
models[2]: shelves1
models[3]: shelves2
models[4]: ground


In [3]:
seed_points = benchmark_helpers.load_seed_points(TEST_SCENE)
end_effector_model_info = models[1]
print("End Effector Bodies:")
for body_index in plant.GetBodyIndices(end_effector_model_info.model_instance):
    print(plant.get_body(body_index).name())
end_effector = plant.GetBodyByName("body", end_effector_model_info.model_instance)
end_effector_frame_id = plant.GetBodyFrameIdOrThrow(end_effector.index())

for seed_point in seed_points:
    plant.SetPositions(plant_context, seed_point)
    diagram.ForcedPublish(diagram_context)
    # time.sleep(0.5)

shelf_seed_point_indices = list(range(1,6))
shelf_seed_points = seed_points[shelf_seed_point_indices]
    

End Effector Bodies:
body
left_finger
right_finger


In [4]:
seed_point_index = 4
with open(
    os.path.join(
        experiment_data_folder, f"rrt_tree_{seed_point_index}_points_and_v_graph.pkl"
    ),
    "rb",
) as f:
    points, visibility_graph, clique_cover = pickle.load(f)

regions = list(LoadIrisRegionsYamlFile(
    os.path.join(experiment_data_folder, f"rrt_tree_{seed_point_index}_regions.pkl")
).values())

In [5]:
regions

In [6]:
for idx, s in enumerate(regions):
    color = Rgba(*utils.hex_to_rgb_0_1(colors[idx]), 1)
    clique_points = points[:, list(clique_cover[idx])]
    region_name = f"seed_point_{seed_point_index}/set_{idx}"
    utils.plot_end_effector_configurations_as_point_cloud(
        clique_points.T,
        end_effector,
        plant,
        meshcat,
        region_name + "/clique_points",
        radius=0.03,
        color=color,
        visible=False,
    )
    utils.visualize_region(
        s,
        end_effector,
        plant,
        meshcat,
        region_name + "/region",
        color=color,
        visible=False,
    )

utils.plot_end_effector_configurations_as_point_cloud(
    points.T,
    end_effector,
    plant,
    meshcat,
    "rrt_points",
    color=Rgba(0, 0, 0, 1),
    visible=False,
)

In [7]:
for i in range(len(clique_cover)-1):
    for j in range(i+1, len(clique_cover)):
        clique0 =points[:, list(clique_cover[i])]
        clique0_vpoly = VPolytope(clique0)
        clique1 =points[:, list(clique_cover[j])]
        clique1_vpoly = VPolytope(clique1)
        
        
        clique_0_point_in_cvx_hull_clique_1 = [
            clique1_vpoly.PointInSet(v) for v in clique0_vpoly.vertices().T
        ]
        print(sum(clique_0_point_in_cvx_hull_clique_1))


2
1
0
2
1
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [8]:
def intersection_over_union(hpoly1, hpoly2, rel_accuracy = 1e-2, max_samples = int(1e4)):
    generator = RandomGenerator(0)
    vol1 = hpoly1.CalcVolumeViaSampling(generator, rel_accuracy, max_samples)
    vol2 = hpoly2.CalcVolumeViaSampling(generator, rel_accuracy, max_samples)
    
    vol_int = hpoly1.Intersection(hpoly2).CalcVolumeViaSampling(generator, rel_accuracy, max_samples)
    
    return vol_int.volume/(vol1.volume+vol2.volume), vol1.volume, vol2.volume, vol_int.volume

iou, vol1, vol2, vol_int = intersection_over_union(regions[0], regions[1], rel_accuracy = 1e-4, max_samples = int(1e6))
print(f"{iou=}")
print(f"{vol1=}")
print(f"{vol2=}")
print(f"{vol_int=}")

iou=0.2185405244629954
vol1=0.026793915574310823
vol2=0.017228832090527724
vol_int=0.009620754362975922


In [9]:
from pydrake.all import PiecewisePolynomial
def random_walk(region, num_steps):
    sampler = HPolyhedronPointSampler(region, 10)
    walk_points = sampler.SamplePoints(num_steps, RandomGenerator())
    traj = PiecewisePolynomial.FirstOrderHold(np.arange(0,num_steps), walk_points)
    return traj

def playback_trajectory(trajectory, discretization_time, trajectory_playback_time = 2):
    """
    discretization time is how finely the trajectory time is discretized
    trajectory_playback_time is how long it will take to playback the whole trajectory.
    """
    diagram_context = diagram.CreateDefaultContext()
    plant_context = plant.GetMyMutableContextFromRoot(diagram_context)
    num_steps = int((trajectory.end_time() - trajectory.start_time()) / discretization_time)
    playback_sleep_dt = trajectory_playback_time/num_steps
    t = trajectory.start_time()
    count = 0
    while t < trajectory.end_time():
        plant.SetPositions(plant_context, trajectory.value(t))
        diagram.ForcedPublish(diagram_context)
        time.sleep(playback_sleep_dt)
        t += discretization_time
        count += 1



    

In [20]:
walk0 = random_walk(regions[0], 100)
playback_trajectory(walk0, 0.05, 2)

In [21]:
walk1 = random_walk(regions[1], 100)
playback_trajectory(walk1, 0.05, 2)

In [23]:
walk_0_1 = random_walk(regions[0].Intersection(regions[1]), 100)
playback_trajectory(walk0, 0.05, 2)

In [13]:
print(f"{regions[0].ChebyshevCenter()=}")
print(f"{regions[1].ChebyshevCenter()=}")
print(f"{regions[0].ChebyshevCenter()-regions[1].ChebyshevCenter()=}")

regions[0].ChebyshevCenter()=array([-2.01933809,  0.91940431, -0.79523759, -0.68098757,  2.37302381,
       -0.99114897,  0.4960834 ])
regions[1].ChebyshevCenter()=array([-1.70165333,  0.9561218 , -1.26505187, -0.98702958,  2.41656976,
       -1.04075095,  0.50088458])
regions[0].ChebyshevCenter()-regions[1].ChebyshevCenter()=array([-0.31768476, -0.03671748,  0.46981428,  0.30604201, -0.04354595,
        0.04960198, -0.00480119])


In [14]:
def get_inscribed_ellipse_axes(region: HPolyhedron):
    ellipse = region.MaximumVolumeInscribedEllipsoid()
    eigs, axes = np.linalg.eigh(ellipse.A())
    center = ellipse.center()
    return eigs, axes, center

eigs0, axes0, center0 = get_inscribed_ellipse_axes(regions[0])
eigs1, axes1, center1 = get_inscribed_ellipse_axes(regions[1])
# Set NumPy print options to format to two decimal places
np.set_printoptions(precision=3)
print(f"{center0=}")
print(f"{center1=}")
print(f"{center1-center0=}")

center0=array([-1.927,  0.928, -0.867, -0.729,  2.301, -1.01 ,  0.582])
center1=array([-1.911,  0.952, -1.021, -0.776,  2.397, -0.948,  0.741])
center1-center0=array([ 0.015,  0.024, -0.155, -0.047,  0.096,  0.062,  0.159])


In [15]:
print(f"{eigs0=}")
print(f"{eigs1=}")
print(f"{eigs0-eigs1=}")

eigs0=array([ 0.928,  1.335,  1.856,  2.234,  4.319,  4.514, 19.954])
eigs1=array([ 1.218,  1.433,  1.931,  2.047,  4.352,  5.485, 21.573])
eigs0-eigs1=array([-0.29 , -0.097, -0.076,  0.187, -0.033, -0.971, -1.619])


In [16]:
print(f"{np.sum(axes0*axes1, axis = 0)=}")
# print(f"{axes1=}")

np.sum(axes0*axes1, axis = 0)=array([ 0.986, -0.969,  0.968, -0.961,  0.939,  0.95 ,  0.996])
